In [42]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [ ]:
doc = pd.read_csv("../../data/raw/starling_doc.tsv", sep = "\t")

doc

In [ ]:
columns_of_interest = ["Character", "Modern (Beijing) reading", "English meaning"]

bc = pd.read_csv("../../data/raw/starling_bigchina.tsv", sep = "\t")

print(
    "Showing what columns have been dropped, in case there is future interest:\n",
    bc[[col for col in bc.columns if col not in columns_of_interest]].head(1).to_dict("records")
)

bc = bc.loc[
    ~bc.Character.isin(["**", "�"]) & (bc.Character.str.len() == 1), columns_of_interest
].dropna().reset_index(drop = True)

bc

In [45]:
import re
import unicodedata

# Pinyin initials (including digraphs)
INITIALS = [
    "zh", "ch", "sh",  # must come before "z", "c", "s"
    "b", "p", "m", "f", "d", "t", "n", "l",
    "g", "k", "h", "j", "q", "x",
    "r", "z", "c", "s", "y", "w"
]

# Vowel mapping with tones
PINYIN_TONE_MAP = {
    "ā": ("a", 1), "á": ("a", 2), "ǎ": ("a", 3), "à": ("a", 4),
    "ē": ("e", 1), "é": ("e", 2), "ě": ("e", 3), "è": ("e", 4),
    "ī": ("i", 1), "í": ("i", 2), "ǐ": ("i", 3), "ì": ("i", 4),
    "ō": ("o", 1), "ó": ("o", 2), "ǒ": ("o", 3), "ò": ("o", 4),
    "ū": ("u", 1), "ú": ("u", 2), "ǔ": ("u", 3), "ù": ("u", 4),
    "ǖ": ("ü", 1), "ǘ": ("ü", 2), "ǚ": ("ü", 3), "ǜ": ("ü", 4),
    # plain vowels (neutral tone)
    "a": ("a", 5), "e": ("e", 5), "i": ("i", 5), "o": ("o", 5),
    "u": ("u", 5), "ü": ("ü", 5),
}

def split_pinyin(pinyin):
    # Normalize to NFC so composed characters (e.g. ī = i + combining macron)
    # match the keys in PINYIN_TONE_MAP
    pinyin = unicodedata.normalize("NFC", pinyin)

    # Step 1: find and normalize tone-marked vowel
    tone = 5
    final_chars = []
    for ch in pinyin:
        if ch in PINYIN_TONE_MAP:
            base, t = PINYIN_TONE_MAP[ch]
            if t != 5:  # don't let a plain vowel overwrite an already-found tone
                tone = t
            final_chars.append(base)
        else:
            final_chars.append(ch)
    pinyin_norm = ''.join(final_chars)

    # Step 2: match initial from the start
    initial = ''
    for init in INITIALS:
        if pinyin_norm.startswith(init):
            initial = init
            break
    final = pinyin_norm[len(initial):]

    return initial, final, tone

bc["decomposed_pinyin"] = bc["Modern (Beijing) reading"].apply(split_pinyin)
bc.Character = bc.Character.str.replace("[", "").str.replace("]", "")

In [46]:
bc

,Character,Modern (Beijing) reading,English meaning,decomposed_pinyin
0,一,yī,"be one, single, whole","(y, i, 1)"
1,乙,yǐ,the 2d of the Heavenly Stems,"(y, i, 3)"
2,丁,dīng,"4th Heavenly Stem; to beat, strike","(d, ing, 1)"
3,七,qī,be seven,"(q, i, 1)"
4,乃,nǎi,"then, and then, now; your (possess.)","(n, ai, 3)"
...,...,...,...,...
5367,虌,biē,an aquatic turtle (Trionyx sinensis),"(b, ie, 1)"
5368,钁,jué,big hoe [Han],"(j, ue, 2)"
5369,驩,huān,= 歡 q. v.,"(h, uan, 1)"
5370,麷,fēng,grain or buds boiled in water,"(f, eng, 1)"


In [ ]:
bc.to_pickle("../../data/cleaned/starling_cleaned.pkl.xz")